<a href="https://colab.research.google.com/github/ailingomezromay/Seminario-LLMs/blob/main/notebook_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Práctico: RAG sobre Legislación Argentina

**Materia:** Aplicaciones con Modelos de Lenguaje (LLMs)
**Opción elegida:** A — RAG (Retrieval-Augmented Generation)
**Alumnos:** *(Ailin Gomez Romay-Vanesa Vidal- Victoria Reppucci-Carlos Lazzarino)*

---

## Objetivo del sistema

Construir un asistente que responda preguntas sobre un corpus de **legislación argentina** (leyes y normas publicadas en fuentes oficiales como InfoLEG y el Boletín Oficial), usando **RAG**: en lugar de que el modelo responda "de memoria", primero **busca** la información en los documentos y luego **redacta** la respuesta en base a lo que encontró.

Al final comparamos las respuestas **CON RAG** vs **SIN RAG** para 5 preguntas.


## ¿Qué es RAG?

**Qué es:** RAG (*Retrieval-Augmented Generation*, "generación aumentada por recuperación") es una técnica que combina dos pasos: primero **recupera** fragmentos relevantes de una base de documentos y después se los pasa al modelo de lenguaje para que **genere** la respuesta usando esa información.

**Para qué sirve:** para que el modelo responda sobre información que no tiene aprendida (nuestros documentos privados o muy específicos) y para **reducir las alucinaciones** (respuestas inventadas), porque lo obligamos a basarse en un texto concreto.

**Analogía:** es la diferencia entre rendir un examen **de memoria** y rendirlo **a libro abierto**. Sin RAG, el modelo contesta con lo que recuerda (y a veces inventa). Con RAG, primero abre el "libro" (nuestros documentos), busca la parte que corresponde y recién ahí responde.

### El pipeline que vamos a construir

```
PDFs  ->  Chunking  ->  Embeddings  ->  Base vectorial (Chroma)
                                                  |
                       Pregunta  ->  busqueda top-k  ->  Contexto + Pregunta  ->  LLM  ->  Respuesta
```

| Paso | Que hace |
|------|----------|
| **Chunking** | Corta los documentos largos en fragmentos ("chunks") manejables. |
| **Embeddings** | Convierte cada chunk en un vector de numeros que representa su significado. |
| **Base vectorial** | Guarda esos vectores y permite buscar por similitud. |
| **Top-k** | Ante una pregunta, recupera los *k* fragmentos mas parecidos. |
| **Generacion** | El LLM redacta la respuesta usando esos fragmentos como contexto. |


## 1. Instalacion de dependencias

Instalamos las librerias del pipeline:
- **chromadb**: base de datos vectorial.
- **sentence-transformers**: modelo de embeddings (multilingue, anda bien en espanol).
- **pypdf**: para leer el texto de los PDFs.
- **google-genai**: cliente oficial de Gemini (el LLM que genera las respuestas).
- **pandas**: para mostrar la comparacion en una tabla.

In [3]:
!pip install -q chromadb sentence-transformers pypdf google-genai pandas
print("Dependencias instaladas.")

Dependencias instaladas.


## 2. Configuracion de la API de Gemini

Necesitas una **API key gratuita** de Gemini. La obtenes en https://aistudio.google.com/apikey (tiene un nivel gratuito suficiente para este TP).

Al ejecutar la celda te va a pedir que la pegues (no queda escrita en el notebook).

In [4]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Ingresa tu GEMINI_API_KEY: ")

Ingresa tu GEMINI_API_KEY: ··········


In [5]:
from google import genai

# Creamos el cliente de Gemini y elegimos el modelo
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO_LLM = "gemini-2.5-flash"

# Prueba rapida para confirmar que la API responde
resp = client.models.generate_content(model=MODELO_LLM, contents="Responde solo con la palabra: OK")
print("Respuesta de Gemini:", resp.text)

Respuesta de Gemini: OK


## 3. Carga del corpus (minimo 5 documentos)

Pone tus PDFs de legislacion (al menos 5, descargados de InfoLEG o el Boletin Oficial) dentro de una carpeta llamada `documentos`.

- **En Google Colab:** ejecuta la celda de subida (descomentada) para subir los archivos.
- **En tu compu / repositorio Git:** crea la carpeta `documentos/` y copia ahi los PDFs.

In [15]:
import os
os.makedirs("documentos", exist_ok=True)

# --- Subir PDFs en Google Colab ---
from google.colab import files
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("documentos", nombre))

print("Archivos en la carpeta 'documentos':")
print(os.listdir("documentos"))

Saving Ley 24.240 — Defensa del Consumidor.pdf to Ley 24.240 — Defensa del Consumidor.pdf
Saving Ley 25.326 — Protección de Datos Personales.pdf to Ley 25.326 — Protección de Datos Personales.pdf
Saving Ley 25.675 — Ley General del Ambiente.pdf to Ley 25.675 — Ley General del Ambiente.pdf
Saving Ley 27.275 — Derecho de Acceso a la Información Pública.pdf to Ley 27.275 — Derecho de Acceso a la Información Pública.pdf
Saving Ley 27.401 — Responsabilidad Penal de Personas Jurídicas.pdf to Ley 27.401 — Responsabilidad Penal de Personas Jurídicas.pdf
Archivos en la carpeta 'documentos':
['Ley 25.675 — Ley General del Ambiente.pdf', 'Ley 24.240 — Defensa del Consumidor.pdf', 'Ley 27.401 — Responsabilidad Penal de Personas Jurídicas.pdf', 'Ley 27.275 — Derecho de Acceso a la Información Pública.pdf', 'Ley 25.326 — Protección de Datos Personales.pdf']


In [16]:
from pypdf import PdfReader

def leer_pdfs(carpeta):
    """Lee todos los PDFs de una carpeta y devuelve su texto + nombre de archivo."""
    documentos = []
    for nombre in sorted(os.listdir(carpeta)):
        if nombre.lower().endswith(".pdf"):
            ruta = os.path.join(carpeta, nombre)
            lector = PdfReader(ruta)
            texto = ""
            for pagina in lector.pages:
                texto += (pagina.extract_text() or "") + "\n"
            documentos.append({"fuente": nombre, "texto": texto})
            print(f"Leido: {nombre}  ({len(texto)} caracteres)")
    return documentos

documentos = leer_pdfs("documentos")
print(f"\nTotal de documentos cargados: {len(documentos)}")
assert len(documentos) >= 5, "El TP pide un minimo de 5 documentos."

Leido: Ley 24.240 — Defensa del Consumidor.pdf  (68551 caracteres)
Leido: Ley 25.326 — Protección de Datos Personales.pdf  (42068 caracteres)
Leido: Ley 25.675 — Ley General del Ambiente.pdf  (37080 caracteres)
Leido: Ley 27.275 — Derecho de Acceso a la Información Pública.pdf  (41361 caracteres)
Leido: Ley 27.401 — Responsabilidad Penal de Personas Jurídicas.pdf  (25101 caracteres)

Total de documentos cargados: 5


## 4. Chunking (division en fragmentos)

**Que es:** cortar cada documento largo en fragmentos mas chicos.

**Para que sirve:** un modelo no procesa bien un texto enorme de una sola vez, y ademas la busqueda es mas precisa si los pedazos son chicos. El **solapamiento** (overlap) hace que dos chunks consecutivos compartan un poco de texto, para no cortar una idea justo a la mitad.

**Analogia:** en vez de resaltar un libro entero, lo dividis en parrafos: despues es mucho mas facil encontrar el parrafo exacto que responde tu pregunta.

In [28]:
def dividir_en_chunks(texto, tam=1500, solapamiento=300):
    """
    Divide el texto en fragmentos de ~'tam' caracteres, tratando de NO
    cortar en medio de una oracion. Cuando llega al limite, retrocede hasta
    el final de oracion mas cercano para que las enumeraciones (a, b, c...)
    no queden partidas al medio.
    """
    texto = " ".join(texto.split())  # limpia espacios y saltos de linea sobrantes
    chunks = []
    inicio = 0
    while inicio < len(texto):
        fin = inicio + tam
        if fin >= len(texto):
            chunk = texto[inicio:].strip()
            if chunk:
                chunks.append(chunk)
            break
        corte = texto.rfind(". ", inicio, fin)      # ultimo fin de oracion antes del limite
        if corte == -1 or corte <= inicio + tam // 2:
            corte = fin                              # si no hay buen corte, cortamos en el limite
        else:
            corte = corte + 1
        chunk = texto[inicio:corte].strip()
        if chunk:
            chunks.append(chunk)
        inicio = max(corte - solapamiento, inicio + 1)  # avanza con solapamiento, sin trabarse

    return chunks

# Aplicamos el chunking a todos los documentos, guardando la fuente de cada chunk
todos_chunks, metadatos, ids = [], [], []
contador = 0
for doc in documentos:
    for chunk in dividir_en_chunks(doc["texto"]):
        todos_chunks.append(chunk)
        metadatos.append({"fuente": doc["fuente"]})
        ids.append(f"chunk_{contador}")
        contador += 1

print(f"Total de chunks generados: {len(todos_chunks)}")
print("\nEjemplo de chunk:\n", todos_chunks[0][:300], "...")

Total de chunks generados: 202

Ejemplo de chunk:
 DEFENSA DEL CONSUMIDOR Ley Nº 24.240 Normas de Protección y Defensa de los Consumidores. Autoridad de Aplicación. Procedimiento y Sanciones. Disposiciones Finales. Sancionada: Setiembre 22 de 1993. Promulgada Parcialmente: Octubre 13 de 1993. Ver Antecedentes Normativos El Senado y Cámara de Diputad ...


## 5. Embeddings y base vectorial (Chroma)

**Que son los embeddings:** convertir texto en un vector de numeros que captura su **significado**. Dos textos que hablan de lo mismo quedan "cerca" en ese espacio numerico, aunque usen palabras distintas.

**Para que sirve la base vectorial:** guarda todos esos vectores y permite buscar, ante una pregunta, cuales son los fragmentos mas parecidos **por significado** (no por coincidencia exacta de palabras).

**Analogia:** es como un mapa donde los textos que tratan temas parecidos quedan agrupados en el mismo barrio. Buscar deja de ser "encontrar la palabra igual" y pasa a ser "ir al barrio correcto".

Usamos el modelo multilingue `paraphrase-multilingual-MiniLM-L12-v2`, que funciona bien en espanol. Chroma calcula los embeddings automaticamente al cargar los chunks.

In [30]:
import chromadb
from chromadb.utils import embedding_functions

# Modelo de embeddings multilingue (la primera vez se descarga, tarda un poco)
funcion_embeddings = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

cliente_chroma = chromadb.Client()

# Si ya existe la coleccion (por re-ejecutar la celda), la borramos para empezar limpio
try:
    cliente_chroma.delete_collection("legislacion")
except Exception:
    pass

coleccion = cliente_chroma.create_collection(
    name="legislacion",
    embedding_function=funcion_embeddings,
)

# Cargamos los chunks: Chroma genera los embeddings y guarda los metadatos (la fuente)
coleccion.add(documents=todos_chunks, metadatas=metadatos, ids=ids)
print(f"Chunks cargados en la base vectorial: {coleccion.count()}")

Chunks cargados en la base vectorial: 202


## 6. Recuperacion (busqueda top-k)

**Que es:** dada una pregunta, buscar en la base vectorial los *k* fragmentos mas parecidos.

**Para que sirve:** son esos fragmentos los que despues le vamos a pasar al LLM como "contexto" para que responda con informacion real de los documentos.

In [31]:
def recuperar(pregunta, k=4):
    """Devuelve los k chunks mas parecidos a la pregunta, con su fuente."""
    resultado = coleccion.query(query_texts=[pregunta], n_results=k)
    chunks = resultado["documents"][0]
    fuentes = [m["fuente"] for m in resultado["metadatas"][0]]
    return chunks, fuentes

# Prueba: cambia la pregunta por algo que este en tus documentos
chunks, fuentes = recuperar("Cual es el objeto de la ley?", k=3)
for i, (c, f) in enumerate(zip(chunks, fuentes), start=1):
    print(f"[{i}] Fuente: {f}\n{c[:250]}...\n")

[1] Fuente: Ley 27.275 — Derecho de Acceso a la Información Pública.pdf
etos obligados enumerados en el artículo 7° de la presente ley, con las únicas limitaciones y excepciones que establece esta norma. Se presume pública toda información que generen, obtengan, transformen, controlen o custodien los sujetos obligados al...

[2] Fuente: Ley 27.401 — Responsabilidad Penal de Personas Jurídicas.pdf
ntervinientes en la comisión del delito; c) Hubiere devuelto el beneficio indebido obtenido. ARTÍCULO 10.- Decomiso. En todos los casos previstos en esta ley serán de aplicación las normas relativas al decomiso establecidas en el Código Penal. ARTÍCU...

[3] Fuente: Ley 27.401 — Responsabilidad Penal de Personas Jurídicas.pdf
erpetua, el que al ser debidamente requerido, no justificare la procedencia de un enriquecimiento patrimonial apreciable suyo o de persona interpuesta para disimularlo, ocurrido con posterioridad a la asunción de un cargo o empleo público y hasta dos...



## 7. Generacion SIN RAG (linea de base)

Le preguntamos directamente al modelo, **sin darle los documentos**. Esto es nuestra base de comparacion: aca el modelo responde solo con lo que "recuerda" de su entrenamiento.

In [32]:
def responder_sin_rag(pregunta):
    """Le pregunta directamente al LLM, sin contexto de los documentos."""
    resp = client.models.generate_content(model=MODELO_LLM, contents=pregunta)
    return resp.text

## 8. Generacion CON RAG

Recuperamos los fragmentos relevantes y armamos un **prompt** que le da al modelo el contexto + la pregunta, con la instruccion de responder **solo** con esa informacion. Si no esta, debe decir que no tiene datos suficientes (asi evitamos que invente).

In [33]:
def responder_con_rag(pregunta, k=4):
    """Recupera contexto de los documentos y se lo pasa al LLM."""
    chunks, fuentes = recuperar(pregunta, k=k)
    contexto = "\n\n".join(chunks)
    prompt = f"""Sos un asistente que responde SOLO con el contexto dado.
Si la respuesta no esta en el contexto, responde: "No tengo informacion suficiente en los documentos".

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}

RESPUESTA:"""
    resp = client.models.generate_content(model=MODELO_LLM, contents=prompt)
    return resp.text, fuentes

## 9. Evaluacion: CON RAG vs SIN RAG

Comparamos las respuestas para 5 preguntas. Lo esperable es que **con RAG** las respuestas sean mas precisas, se ajusten a *tus* documentos e indiquen la fuente, mientras que **sin RAG** el modelo puede ser vago o inventar.

> **Importante:** reemplaza las preguntas por otras que se puedan responder con TUS documentos concretos.

In [35]:
import pandas as pd

preguntas = [
    "¿De cuántos días hábiles dispone el presunto infractor para presentar su descargo en las actuaciones administrativas?",          # Ley 24.240
    "¿Cuál es la autoridad de aplicación de la ley de protección de datos personales?",                                              # Ley 25.326
    "¿A qué delitos se aplica el régimen de responsabilidad penal de las personas jurídicas?",                                       # Ley 27.401
    "¿Qué principios de política ambiental establece la ley general del ambiente?",                                                  # Ley 25.675
    "¿Quiénes son los sujetos obligados a brindar información pública?",                                                             # Ley 27.275
]

filas = []
for p in preguntas:
    print(f"Procesando: {p}")
    sin_rag = responder_sin_rag(p)
    con_rag, fuentes = responder_con_rag(p)
    filas.append({
        "Pregunta": p,
        "Respuesta SIN RAG": sin_rag,
        "Respuesta CON RAG": con_rag,
        "Fuentes recuperadas": ", ".join(sorted(set(fuentes))),
    })

df = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", None)
df

Procesando: ¿De cuántos días hábiles dispone el presunto infractor para presentar su descargo en las actuaciones administrativas?
Procesando: ¿Cuál es la autoridad de aplicación de la ley de protección de datos personales?
Procesando: ¿A qué delitos se aplica el régimen de responsabilidad penal de las personas jurídicas?


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 13.639393532s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

### Ver una comparacion en detalle (opcional)

Ejecuta esta celda para leer una pregunta puntual de forma mas comoda.

In [ ]:
i = 4  # cambia el indice (0 a 4) para ver otra pregunta

print("PREGUNTA:", filas[i]["Pregunta"])
print("\n--- SIN RAG ---\n", filas[i]["Respuesta SIN RAG"])
print("\n--- CON RAG ---\n", filas[i]["Respuesta CON RAG"])
print("\nFuentes:", filas[i]["Fuentes recuperadas"])

## 10. Conclusiones y analisis critico

Análisis crítico


Al comparar las respuestas con y sin RAG en las cinco preguntas, no encontramos que un método fuera siempre mejor que el otro, sino que cada uno falla y acierta de manera distinta.

La ventaja más clara del RAG apareció en las preguntas sobre leyes nacionales específicas. Sin RAG, el modelo respondió con seguridad pero sobre el país equivocado: al preguntar por los delitos de las personas jurídicas contestó con legislac## Resultados de la evaluación: CON RAG vs SIN RAG

| # | Pregunta (Ley) | Respuesta SIN RAG | Respuesta CON RAG | Fuente recuperada |
|---|----------------|-------------------|-------------------|-------------------|
| 0 | Plazo de descargo (Ley 24.240) | Vaga: da un rango amplio de días para distintos países | Exacta: "cinco días hábiles", tomado del texto | Con ruido: recuperó 2 leyes |
| 1 | Autoridad de aplicación de datos personales (Ley 25.326) | Correcta y completa: la AAIP | Vaga: "el organismo de control" (el dato no está en la ley, que es de 2000) | Limpia |
| 2 | Delitos de personas jurídicas (Ley 27.401) | Detallada pero de **España** (jurisdicción equivocada) | Argentina y correcta, pero **incompleta** (solo un delito de la lista) | Limpia |
| 3 | Principios ambientales (Ley 25.675) | Detallada pero de **El Salvador** (jurisdicción equivocada) | Correcta y completa: congruencia, prevención, precautorio, equidad intergeneracional, progresividad | Limpia |
| 4 | Sujetos obligados a dar información (Ley 27.275) | Genérica y sin fuente concreta | Argentina y correcta, pero **incompleta** (solo el primer inciso) | Limpia |ión de España, y al preguntar por los principios ambientales respondió con la ley de El Salvador. Con RAG, en cambio, las respuestas se mantuvieron ancladas a la ley argentina correcta, porque el modelo solo podía usar los documentos que le di. También se vio la precisión del RAG en la pregunta por el plazo de descargo: sin RAG dio un rango vago de días, y con RAG dio el dato exacto (cinco días hábiles) tomado del texto.


Sin embargo, el RAG también mostró limitaciones importantes:

Respuestas incompletas por el chunking. En las preguntas por los delitos (Ley 27.401) y por los sujetos obligados (Ley 27.275), el sistema recuperó solo una parte de la lista, porque el fragmento cortó la enumeración por la mitad. La respuesta fue correcta pero parcial.
RAG no puede responder lo que no está en el documento. En la pregunta por la autoridad de aplicación de datos personales, el RAG respondió apenas "el organismo de control", mientras que el modelo sin RAG sí supo que hoy es la AAIP. Esto pasa porque la Ley 25.326 es del año 2000 y la AAIP se creó en 2017: el dato es más nuevo que la ley y no está en el texto cargado.
Ruido en la recuperación. En la primera pregunta, el sistema trajo fragmentos de dos leyes distintas cuando la respuesta salía de una sola. La búsqueda por similitud a veces recupera texto parecido pero no pertinente.

Conclusión:
 El RAG es muy útil para anclar las respuestas a fuentes concretas y evitar que el modelo invente o confunda jurisdicciones, pero no es infalible: su calidad depende de que la información esté completa y presente en el corpus. Como mejoras, probaría ajustar el tamaño de los chunks y el solapamiento para no cortar las listas, subir o bajar el valor de k según el caso, agregar la cita del artículo en cada respuesta, y mantener el corpus actualizado para evitar el problema de los datos desactualizados.